# Data Preparation
## Credit Risk Scorecard Project

Following CRISP-DM Phase 3. This notebook:
1. Cleans the main application table
2. Engineers domain-aware features
3. Aggregates bureau and previous application tables
4. Produces a single clean modelling dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully")

In [ ]:
# Load main table
app = pd.read_csv('data/home-credit-default-risk/application_train.csv')
print(f"Application train shape: {app.shape}")
print(f"Default rate: {app['TARGET'].mean():.2%}")

In [ ]:
# DAYS columns are negative (days before application)
# Convert to positive and create age in years for interpretability

app['DAYS_BIRTH'] = abs(app['DAYS_BIRTH'])
app['DAYS_EMPLOYED'] = abs(app['DAYS_EMPLOYED'])
app['DAYS_REGISTRATION'] = abs(app['DAYS_REGISTRATION'])
app['DAYS_ID_PUBLISH'] = abs(app['DAYS_ID_PUBLISH'])

# DAYS_EMPLOYED has an anomaly - value of 365243 means pensioner/unemployed
# Replace with NaN
# Keep a flag BEFORE replacing, so the model can see this group
app['DAYS_EMPLOYED_ANOM'] = (app['DAYS_EMPLOYED'] == 365243).astype(int)
app['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

# Create age in years (more interpretable)
app['AGE_YEARS'] = app['DAYS_BIRTH'] / 365

print("DAYS columns cleaned")
print(f"Age range: {app['AGE_YEARS'].min():.0f} to {app['AGE_YEARS'].max():.0f} years")
print(f"DAYS_EMPLOYED anomalies replaced: {app['DAYS_EMPLOYED'].isna().sum():,}")

In [ ]:
# Feature engineering - domain aware financial ratios

# Credit to income ratio: how large is the loan relative to their income?
app['CREDIT_INCOME_RATIO'] = app['AMT_CREDIT'] / app['AMT_INCOME_TOTAL']

# Annuity to income ratio: can they afford the monthly payments?
app['ANNUITY_INCOME_RATIO'] = app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL']

# Credit to goods price ratio: are they borrowing more than the goods cost?
app['CREDIT_GOODS_RATIO'] = app['AMT_CREDIT'] / app['AMT_GOODS_PRICE']

# Income per family member
app['INCOME_PER_PERSON'] = app['AMT_INCOME_TOTAL'] / app['CNT_FAM_MEMBERS'].clip(lower=1)

# Employment length as fraction of age: how stable is their employment history?
app['EMPLOYED_TO_AGE_RATIO'] = app['DAYS_EMPLOYED'] / app['DAYS_BIRTH']

print("Feature engineering complete")
print(f"Total columns: {app.shape[1]}")

# average credit to income ratio by default status
print("\nAverage credit/income ratio by default status:")
print(app.groupby('TARGET')['CREDIT_INCOME_RATIO'].mean().round(2))

In [ ]:
# Separate columns by type.
# NOTE: numeric imputation is NOT done here. It is done after the train/test
# split (fitted on train only) to avoid data leakage.
categorical_cols = app.select_dtypes(include='object').columns.tolist()
numerical_cols = app.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols = [c for c in numerical_cols if c != 'TARGET']

# Categorical gaps get an explicit 'Missing' label (uses no statistics, so no leakage)
for col in categorical_cols:
    app[col] = app[col].fillna('Missing')

print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numerical columns: {len(numerical_cols)}")
print(f"Numeric missing values left for post-split imputation: {app[numerical_cols].isnull().sum().sum():,}")

In [ ]:
# Encode categorical columns
# Use Label Encoding for binary categories (2 unique values)
# Use One-Hot Encoding for multi-category columns

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Label encode binary categoricals
binary_cols = [col for col in categorical_cols 
               if app[col].nunique() == 2]

for col in binary_cols:
    app[col] = le.fit_transform(app[col].astype(str))

print(f"Label encoded {len(binary_cols)} binary columns")

# One-hot encode remaining categoricals
multi_cols = [col for col in categorical_cols 
              if app[col].nunique() > 2]

app = pd.get_dummies(app, columns=multi_cols)

print(f"One-hot encoded {len(multi_cols)} multi-category columns")
print(f"Total columns after encoding: {app.shape[1]}")

In [ ]:
# Load and aggregate bureau table
bureau = pd.read_csv('data/home-credit-default-risk/bureau.csv')
print(f"Bureau shape: {bureau.shape}")
print(f"Unique applicants in bureau: {bureau['SK_ID_CURR'].nunique():,}")
print(f"Average bureau records per applicant: {len(bureau)/bureau['SK_ID_CURR'].nunique():.1f}")

In [ ]:
# Aggregate bureau table to one row per applicant
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    # How many previous loans at other banks?
    BUREAU_LOAN_COUNT = ('SK_ID_BUREAU', 'count'),
    
    # How many different loan types?
    BUREAU_LOAN_TYPES = ('CREDIT_TYPE', 'nunique'),
    
    # Average, max current debt outstanding
    BUREAU_DEBT_MEAN = ('AMT_CREDIT_SUM_DEBT', 'mean'),
    BUREAU_DEBT_MAX = ('AMT_CREDIT_SUM_DEBT', 'max'),
    
    # Average, max credit limit across all bureau loans
    BUREAU_CREDIT_MEAN = ('AMT_CREDIT_SUM', 'mean'),
    BUREAU_CREDIT_MAX = ('AMT_CREDIT_SUM', 'max'),
    
    # How many days since their most recent and oldest bureau loan
    BUREAU_DAYS_CREDIT_MEAN = ('DAYS_CREDIT', 'mean'),
    BUREAU_DAYS_CREDIT_MIN = ('DAYS_CREDIT', 'min'),
    
    # How many loans were overdue?
    BUREAU_OVERDUE_MEAN = ('AMT_CREDIT_SUM_OVERDUE', 'mean'),
    
    # How many active vs closed loans?
    BUREAU_ACTIVE_COUNT = ('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    BUREAU_CLOSED_COUNT = ('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),
).reset_index()

# Fill missing values with 0 (applicants with no bureau history)
bureau_agg = bureau_agg.fillna(0)

print(f"Bureau aggregated shape: {bureau_agg.shape}")
print(f"Sample of aggregated features:")
bureau_agg.head(3)

In [ ]:
# Join bureau aggregations onto main table
app = app.merge(bureau_agg, on='SK_ID_CURR', how='left')

# Applicants with no bureau history get 0 for all bureau features
bureau_cols = [col for col in bureau_agg.columns if col != 'SK_ID_CURR']
app[bureau_cols] = app[bureau_cols].fillna(0)

print(f"Shape after joining bureau features: {app.shape}")
print(f"New bureau features added: {len(bureau_cols)}")

In [ ]:
# Load and aggregate previous application table
prev = pd.read_csv('data/home-credit-default-risk/previous_application.csv')
print(f"Previous application shape: {prev.shape}")
print(f"Unique applicants: {prev['SK_ID_CURR'].nunique():,}")
print(f"Average previous applications per applicant: {len(prev)/prev['SK_ID_CURR'].nunique():.1f}")


In [ ]:
# Aggregate previous application table
prev_agg = prev.groupby('SK_ID_CURR').agg(
    # How many previous applications?
    PREV_APP_COUNT = ('SK_ID_PREV', 'count'),
    
    # How many were approved, refused, cancelled?
    PREV_APPROVED_COUNT = ('NAME_CONTRACT_STATUS', 
                           lambda x: (x == 'Approved').sum()),
    PREV_REFUSED_COUNT = ('NAME_CONTRACT_STATUS', 
                          lambda x: (x == 'Refused').sum()),
    PREV_CANCELLED_COUNT = ('NAME_CONTRACT_STATUS', 
                            lambda x: (x == 'Canceled').sum()),
    
    # Average loan amount applied for previously
    PREV_AMT_APPLICATION_MEAN = ('AMT_APPLICATION', 'mean'),
    PREV_AMT_APPLICATION_MAX = ('AMT_APPLICATION', 'max'),
    
    # Average loan amount actually approved
    PREV_AMT_CREDIT_MEAN = ('AMT_CREDIT', 'mean'),
    
    # Average down payment
    PREV_DOWN_PAYMENT_MEAN = ('AMT_DOWN_PAYMENT', 'mean'),
    
    # Days since last application
    PREV_DAYS_DECISION_MIN = ('DAYS_DECISION', 'min'),
    PREV_DAYS_DECISION_MEAN = ('DAYS_DECISION', 'mean'),
).reset_index()

prev_agg = prev_agg.fillna(0)

print(f"Previous application aggregated shape: {prev_agg.shape}")
prev_agg.head(3)

In [ ]:
# Join previous application aggregations onto main table
app = app.merge(prev_agg, on='SK_ID_CURR', how='left')

# Fill NaN for applicants with no previous applications
prev_cols = [col for col in prev_agg.columns if col != 'SK_ID_CURR']
app[prev_cols] = app[prev_cols].fillna(0)

print(f"Shape after joining previous application features: {app.shape}")
print(f"New previous application features added: {len(prev_cols)}")

In [ ]:
# Save cleaned dataset
os.makedirs('data/processed', exist_ok=True)
app.to_csv('data/processed/app_clean.csv', index=False)

print(f"Clean dataset saved successfully")
print(f"Final shape: {app.shape}")
print(f"Target distribution:")
print(app['TARGET'].value_counts())
print(f"\nDefault rate: {app['TARGET'].mean():.2%}")